# v8 捕获范围精确说明

原报告“69 份代码重新静态检查和真实浏览器捕获”描述的是语法优先的整体流程，不应理解为每份都有浏览器截图。精确范围为：

- 69 份有代码产物重新评估。
- 66 份基线保存真实浏览器捕获。
- 3 份脚本语法失败，按生产语法优先门停止，没有浏览器捕获。
- TaskManager 无完整代码，不重试未知调用，也没有浏览器捕获。

35/70 的有效离线结果及全部原始证据、哈希、失败不变。本说明追加保存，不覆盖原报告、Notebook 或原始 SHA256SUMS。`analysis_verified.ipynb` 在原 Notebook 前加本说明，供后续阅读使用。


# 通用修复 v8：生产规则的顺序、初始化及原生控件闭环

2026-11-20。本轮只使用已有冻结源码与 San 产物、真实解析器和浏览器；无真实模型调用、云数据库访问、服务重启或新任务提交。

## 版本与结果

- 首轮混合 70 项：12/70；由 v1 的 49 项和 v2 补充 21 项构成，配置不同，不包装成统一单版本基线。
- v3 真实模型批次仍为 15/70，本轮没有改写这个结果。
- 上一轮 v7 的有效离线结果 33/70；本轮对 69 份有完整代码的有效父版本重新静态检查和真实浏览器捕获，当前基线 30/70，顺序规则后 35/70。TaskManager 无完整产物，不重试未知调用。
- 相对 v7 新增离线通过：AudioSegmentMarker、CrisisCommunicationHub；原通过当前未通过：无。
- 仍有 35 项未闭合，不声称所有问题已彻底解决。关联回归 225 项在固定版本下通过；初始并行运行有一模块超时，保留原失败日志，再在回放结束后单独复测并合并同版证据，不伪装成首次全绿。独立副本验证另存。
- 7166 个历史文件及 411 个 v7 引用文件哈希不变。原始报告、失败和被拒候选均保留。

## 原因定位及修复链

1. 生成代码确有初始化时序缺陷，不是 Vue 非法：San computed 比 inited 更早。源 data 克隆 props 再派生选择状态，不能用 null 集合延迟到 inited。新规则只在纯源 getter／helper、完整初始化表达式及顺序可证明时恢复；保留 JSON 深克隆／浅克隆语义。未知副作用、created 合并、内置对象被重定义时拒绝，不随意给 getter 加 ||[]。
2. 已有规则和候选门没有形成执行顺序闭环：缺省值恢复夹杂模板改动时，不满足既有整份代码窄证明。现在先提出完全默认值规则产物、真实复测和正常门禁，再对接受的版本继续通用规则；被拒则保留父版本，不重复同一提案。恢复运行不等于结构通过。
3. 动态赋值证明缺跨方法参数流：不识别方法名，而是递归追踪所有真实 this 调用及模板入口，要求有限普通 data 键、全部上游调用体和完整模板实参匹配。未知外部键、逃逸、循环、自定义实例命名空间仍不证明。
4. 原生 select 的 value 设置阶段不同：San 在 option 挂载前设置，Vue 在其后。以源码一致的简单绑定，添加可审计的公共 ref＋attached/updated 同步，不增加 DOM 包装、不使用私有 API。语法、静态契约及评估共用同一精确 helper 证明；仅逻辑源比较识别内部前缀，真实 DOM／value／选项更新照常检查。
5. 静态分析器异常不再返回空 findings 当成功：明确保留 source_semantic_analysis_unavailable。生成与修复共享 v5 初始化约束，相关新诊断进入定向修复及按错误选择 SSM 的路径。

本轮不是按组件名称补丁，也不是整体 Vue AST 规则生成。完整 SSM＋模型生成、质量阈值、修复预算及回退门继续保留。

## 进一步定位但尚未修复的校验／规则缺口

AccessMatrix 的纯初始化规则已恢复候选 runtime，但组合了 class 反射且出现控件位置诊断，不能套用只证明默认值规则的窄恢复门，因而仍回退；还需将初始化独立提案并绑定其完整源证明。VolumeControl 已有合法的字符串键生命周期方法，但旧生命周期正则不识别；SearchClearField 的合法双向绑定内部空格使原生 value 规则漏匹配；FileDropIndicator 是 Vue/San class 字符串 trim 差异；StatusSwitch 则确实多了源中没有的 aria-label。这些不能全部归为模型随机或 Vue 源码有错。详见 residual_root_cause_audit.json。这里只登记定位事实，不提前标记已经修好。

## 全部剩余问题

以下计数按组件内诊断去重，不是人工独立缺陷数；分类可重叠。所有剩余项、实际维度、候选门原因和文件路径在 remaining.json，首轮逐项配对在 comparison.json。

- rendered_dom_mismatch: 24
- template_structure_mismatch: 12
- rendered_control_state_mismatch: 7
- source_dynamic_assignment_unproven: 6
- dom_child_order_mismatch: 5
- nested_state_mutation: 4
- target_browser_error: 4
- target_script_syntax_error: 3
- missing_prop_defaults: 3
- missing_event_handlers: 3
- rendered_control_position_mismatch: 3
- missing_lifecycle_mapping: 2
- hidden_template_reactive_dependency: 2
- source_class_binding_not_landed: 2
- source_render_dependency_not_landed: 2
- watch_registration_not_in_lifecycle: 2
- source_data_initialization_phase_not_landed: 2
- custom_event_not_fire: 1
- source_method_interface_not_landed: 1
- invalid_san_template_data_store_access: 1
- model_response_unknown_no_automatic_retry: 1
- vue_this_state: 1
- duplicate_class_attribute: 1
- invalid_san_computed_instance_method_call: 1
- missing_init_data: 1
- unsupported_san_inline_template_math_global: 1
- event_arguments_not_landed: 1
- missing_event_bindings: 1
- missing_watch_migration: 1
- source_dom_event_not_landed: 1
- source_computed_collection_not_landed: 1
- source_filter_method_not_landed: 1
- source_record_update_not_landed: 1
- source_shared_record_projection_not_landed: 1
- source_computed_option_not_landed: 1

原生控件／模板真实缺失、未证明的业务语义、语法损坏和无完整模型产物必须分别继续处理，不能降低阈值或把一次模型重跑当作规则已闭合的证明。

## 证据

- ../composed_v8_final_replay_20261120/：不可变协议、每项基线／步骤／有效代码、真实 DOM 与截图、门禁和实现快照。
- migration_system/docs/verification/2026-11-19_general_repairs_v1/test-results-composed_v8_verified.json：固定版本关联回归；final.json 保留初始并行超时，isolated 日志保留隔离复测。
- migration_system/var/refactor-verification/composed_v8_20261120.json：仓库外独立运行；模型为明确记录响应替身，不是真实生产成功率。
- ../composed_v8_sequential_replay_20261120/STOPPED.md：补充内置对象遮蔽反例前的中止预检，不纳入本次统计。


In [ ]:
import json
from pathlib import Path
summary=json.loads(Path('summary.json').read_text(encoding='utf-8'))
summary

In [ ]:
remaining=json.loads(Path('remaining.json').read_text(encoding='utf-8'))
remaining